<div style="background: linear-gradient(135deg, #1F497D 0%, #2E75B6 100%); padding: 30px; border-radius: 12px; color: white; text-align: center;">
  <h1 style="margin:0; font-size: 2em;">🌸 Лабораторная работа №1</h1>
  <h2 style="margin:10px 0 0 0; font-weight: normal;">Разведочный анализ данных · Исследование и визуализация данных</h2>
  <p style="margin-top: 12px; opacity: 0.85;">Датасет: <b>Iris (Ирисы Фишера)</b> · Инструмент: Python, Pandas, Matplotlib, Seaborn</p>
</div>

---
## ⚙️ 0. Установка и импорт библиотек

Прежде чем начать работу, необходимо подключить все нужные библиотеки.

| Библиотека | Назначение |
|---|---|
| **numpy** | Математические операции с массивами чисел |
| **pandas** | Работа с таблицами данных (DataFrame) |
| **matplotlib** | Базовая библиотека для построения графиков |
| **seaborn** | Высокоуровневая визуализация на основе matplotlib |
| **sklearn** | Машинное обучение; здесь используем только готовый датасет |

In [ ]:
# ── Установка библиотек (выполняется один раз при запуске в Colab) ──────────
# Символ ! означает команду операционной системы (не Python)
!pip install -q seaborn --upgrade

In [ ]:
# ── Импорт библиотек ─────────────────────────────────────────────────────────
import numpy as np                        # массивы и математика
import pandas as pd                       # таблицы данных
import matplotlib.pyplot as plt          # рисование графиков
import matplotlib.gridspec as gridspec   # гибкая сетка для subplots
import seaborn as sns                    # красивые статистические графики

from sklearn.datasets import load_iris  # встроенный датасет
from pandas.plotting import parallel_coordinates  # параллельные координаты

# ── Глобальные настройки отображения ─────────────────────────────────────────
# Разрешение графиков — чем выше, тем чётче изображение
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11

# Ограничиваем число знаков после запятой в таблицах
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Единая цветовая палитра для трёх классов: синий / оранжевый / зелёный
PALETTE = ['#4C72B0', '#DD8452', '#55A868']

print('✅ Все библиотеки успешно импортированы')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')

---
## 📖 1. Описание набора данных

### О датасете Iris

**Iris** — один из самых известных наборов данных в истории машинного обучения и статистики.
Впервые опубликован британским статистиком **Рональдом Фишером** в 1936 году
в статье *«The use of multiple measurements in taxonomic problems»*.

Датасет описывает три вида ирисов:

| Класс | Вид | Объектов |
|---|---|---|
| 0 | 🌸 *Iris setosa* | 50 |
| 1 | 🌺 *Iris versicolor* | 50 |
| 2 | 🌼 *Iris virginica* | 50 |

### Признаки

| Признак | Единица | Что это |
|---|---|---|
| `sepal_length` | см | Длина чашелистика (внешние зелёные лепестки) |
| `sepal_width`  | см | Ширина чашелистика |
| `petal_length` | см | Длина лепестка (цветные внутренние лепестки) |
| `petal_width`  | см | Ширина лепестка |

> **Почему этот датасет?** Он небольшой (150 строк), без пропусков, хорошо изучен
> и идеально подходит для демонстрации методов EDA.

In [ ]:
# ── Загрузка датасета из sklearn ──────────────────────────────────────────────
# as_frame=True возвращает данные уже в виде pandas DataFrame, а не numpy-массива
iris = load_iris(as_frame=True)

# Собираем итоговый DataFrame:
#   iris.frame содержит и признаки, и целевую переменную в одной таблице
df = iris.frame.copy()

# Переименовываем столбцы для удобства (убираем пробелы и единицы измерения)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']

# Добавляем текстовое название вида (вместо чисел 0, 1, 2)
# .map() — применяет словарь-маппинг к каждому значению столбца
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Список числовых признаков (будем часто обращаться к ним)
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# Красивые подписи для осей графиков
feature_labels = {
    'sepal_length': 'Длина чашелистика (см)',
    'sepal_width':  'Ширина чашелистика (см)',
    'petal_length': 'Длина лепестка (см)',
    'petal_width':  'Ширина лепестка (см)'
}

print(f'Датасет загружен. Размер: {df.shape[0]} строк × {df.shape[1]} столбцов')
print()
print('Первые 10 строк:')
df.head(10)

---
## 📊 2. Основные характеристики датасета

На этом этапе мы изучаем **«паспорт»** датасета:
- Размер и структура
- Типы данных
- Наличие пропусков
- Описательная статистика

In [ ]:
# ── 2.1 Форма и типы данных ──────────────────────────────────────────────────
print('━━━ РАЗМЕР ДАТАСЕТА ━━━')
# .shape возвращает кортеж (строки, столбцы)
print(f'  Строки (наблюдения) : {df.shape[0]}')
print(f'  Столбцы (признаки)  : {df.shape[1]}')

print()
print('━━━ ТИПЫ ДАННЫХ ━━━')
# float64 — вещественное число (с дробной частью)
# int64   — целое число
# object  — строка / текст
print(df.dtypes)

print()
print('━━━ ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ ━━━')
# .isnull() возвращает True/False для каждой ячейки
# .sum() считает количество True (= количество пропусков) по каждому столбцу
missing = df.isnull().sum()
print(missing)
print(f'\n  ИТОГО пропусков: {missing.sum()} ← пропусков нет, данные чистые ✅')

In [ ]:
# ── 2.2 Описательная статистика ───────────────────────────────────────────────
# .describe() — ключевой метод pandas, вычисляет 8 статистик по каждому числовому столбцу:
#   count  — количество непустых значений
#   mean   — среднее арифметическое
#   std    — стандартное отклонение (разброс данных)
#   min    — минимум
#   25%    — первый квартиль: 25% значений ниже этого числа
#   50%    — медиана: половина значений ниже, половина выше
#   75%    — третий квартиль: 75% значений ниже этого числа
#   max    — максимум
print('Описательная статистика всех числовых признаков:')
print('(транспонируем .T для удобного чтения — признаки по строкам)')
df[features].describe().T

In [ ]:
# ── 2.3 Статистика по классам ─────────────────────────────────────────────────
# .groupby() разбивает DataFrame на группы по уникальным значениям столбца 'species'
# .mean()    вычисляет среднее значение каждого признака внутри каждой группы
# .round(2)  округляет до 2 знаков после запятой
print('Среднее значение признаков по каждому виду ириса:')
print('(это хорошо показывает, чем виды отличаются друг от друга)')
df.groupby('species')[features].mean().round(2)

In [ ]:
# ── 2.4 Распределение классов ─────────────────────────────────────────────────
# .value_counts() считает количество каждого уникального значения в столбце
print('Количество наблюдений по видам:')
print(df['species'].value_counts())
print()
print('Датасет СБАЛАНСИРОВАН: каждый класс представлен ровно 50 объектами.')
print('Это важно: несбалансированные классы требуют специальных методов обработки.')

---
## 📈 3. Визуальное исследование датасета

Числа говорят не всё. Графики помогают **увидеть** закономерности, которые сложно заметить в таблицах.

### 3.1 Распределение классов

Первое, что нужно проверить — **сколько объектов каждого класса** в датасете.
Несбалансированные классы — частая проблема в ML, требующая специального подхода.

In [ ]:
# ── Круговая диаграмма + столбчатый график ────────────────────────────────────
# plt.subplots(1, 2) — создаём сетку 1 строка × 2 столбца
# figsize=(12, 5)    — размер фигуры в дюймах
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Распределение классов в датасете Iris', fontsize=15, fontweight='bold', y=1.02)

counts = df['species'].value_counts()

# ─ Левый график: круговая диаграмма ─
# autopct='%1.1f%%' — показывает процент с 1 знаком после точки
# startangle=90     — начинаем рисовать с верхушки
# wedgeprops        — внешний вид секторов
wedges, texts, autotexts = axes[0].pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=PALETTE,
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5}
)
for at in autotexts:     # делаем проценты жирными
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[0].set_title('Доля каждого вида', pad=12)

# ─ Правый график: столбчатая диаграмма ─
bars = axes[1].bar(
    counts.index, counts.values,
    color=PALETTE, edgecolor='white', linewidth=1.5, width=0.6
)
# Подписываем число над каждым столбцом
for bar, val in zip(bars, counts.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,   # x: центр столбца
        bar.get_height() + 0.5,               # y: чуть выше столбца
        str(val),
        ha='center', va='bottom', fontweight='bold', fontsize=12
    )

axes[1].set_xlabel('Вид ириса', labelpad=8)
axes[1].set_ylabel('Количество наблюдений', labelpad=8)
axes[1].set_title('Количество объектов по классам', pad=12)
axes[1].set_ylim(0, 65)
axes[1].spines[['top', 'right']].set_visible(False)   # убираем лишние оси
axes[1].grid(axis='y', alpha=0.35, linestyle='--')

plt.tight_layout()
plt.show()
print('\n📌 Вывод: датасет идеально сбалансирован — по 50 объектов каждого вида (33.3%)')

### 3.2 Гистограммы распределения признаков

**Гистограмма** — базовый инструмент EDA. Показывает, как часто встречаются разные значения признака.

- Ось X — значения признака
- Ось Y — сколько наблюдений попало в каждый «ящик» (bin)
- Вертикальная пунктирная линия — среднее значение по всему датасету

In [ ]:
# ── Гистограммы по каждому признаку, разбитые по видам ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))   # сетка 2×2
fig.suptitle('Гистограммы распределения признаков по видам', fontsize=15, fontweight='bold')
axes = axes.flatten()   # превращаем матрицу 2×2 в список из 4 элементов

for ax, feat in zip(axes, features):
    # Для каждого вида рисуем свою гистограмму с прозрачностью (alpha=0.65)
    for color, species in zip(PALETTE, df['species'].unique()):
        subset = df[df['species'] == species][feat]   # данные только этого вида
        ax.hist(
            subset,
            bins=15,             # количество «корзин» (интервалов)
            alpha=0.65,          # прозрачность — чтобы гистограммы не перекрывали друг друга
            color=color,
            label=species,
            edgecolor='white'   # белые границы между столбцами
        )

    # Вертикальная линия = общее среднее по всем видам
    ax.axvline(
        df[feat].mean(),
        color='black', linestyle='--', linewidth=1.8,
        label=f'Среднее = {df[feat].mean():.2f}'
    )

    ax.set_xlabel(feature_labels[feat], labelpad=8)
    ax.set_ylabel('Частота (кол-во наблюдений)', labelpad=8)
    ax.set_title(feature_labels[feat], pad=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print('''
📌 Выводы:
  • sepal_length / sepal_width: распределения близки к нормальному (один горб-колокол).
  • petal_length / petal_width: ДВУГОРБОЕ (бимодальное) распределение!
    Левый горб — Iris setosa (маленькие лепестки), правый — versicolor и virginica.
  → Это говорит, что лепестки — ключевые признаки для разделения классов.
''')

### 3.3 Ящики с усами (Box plots)

**Box plot** (ящик с усами) — компактный способ показать 5 характеристик за один раз:

```
  ──── максимум (усы)
  │
  ┌──────┐  ←  Q3 (75% данных ниже)
  │      │
  │ ═══  │  ←  медиана (50%)
  │      │
  └──────┘  ←  Q1 (25% данных ниже)
  │
  ──── минимум (усы)
  ●          ←  выброс (outlier)
```

Точки за пределами «усов» — **выбросы** (значения, аномально далёкие от основной массы).

In [ ]:
# ── Box plots с наложенными точками (jitter) ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Ящики с усами (Box plots) + отдельные наблюдения', fontsize=15, fontweight='bold')
axes = axes.flatten()

for ax, feat in zip(axes, features):
    # Готовим данные: список из трёх массивов (по одному на вид)
    data_by_species = [
        df[df['species'] == s][feat].values
        for s in ['setosa', 'versicolor', 'virginica']
    ]

    # patch_artist=True — закрашиваем ящики
    # notch=True        — делаем вырез у медианы (показывает доверительный интервал)
    bp = ax.boxplot(
        data_by_species,
        labels=['setosa', 'versicolor', 'virginica'],
        patch_artist=True,
        notch=True,
        whiskerprops={'linewidth': 1.8},
        medianprops={'color': 'black', 'linewidth': 2.5},
        flierprops={'markerfacecolor': 'grey', 'marker': 'o', 'markersize': 6, 'alpha': 0.6}
    )
    # Закрашиваем каждый ящик своим цветом
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    # Накладываем индивидуальные точки с небольшим случайным смещением (jitter)
    # чтобы точки не сливались в одну линию
    for i, (data, color) in enumerate(zip(data_by_species, PALETTE), start=1):
        # np.random.uniform — случайное число в диапазоне [-0.12, 0.12]
        jitter = np.random.uniform(-0.12, 0.12, len(data))
        ax.scatter(
            np.full(len(data), i) + jitter,   # позиция по X: номер группы + смещение
            data,                              # значения по Y
            color=color, alpha=0.35, s=18, zorder=3
        )

    ax.set_ylabel(feature_labels[feat], labelpad=8)
    ax.set_title(feature_labels[feat], pad=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.35, linestyle='--')

plt.tight_layout()
plt.show()

print('''
📌 Выводы:
  • Iris setosa имеет значительно меньшие лепестки — ящик не перекрывается с другими.
  • Для признаков чашелистика ящики перекрываются — виды сложнее разделить.
  • Несколько выбросов (точки вне усов) видны у sepal_width — это реальные редкие значения.
''')

### 3.4 Violin plots — форма распределения + квартили

**Violin plot** = Box plot + форма распределения (KDE).

Ширина «скрипки» в каждой точке показывает, **сколько наблюдений** имеют такое значение:
- Широкое место → много наблюдений с таким значением
- Узкое место → мало наблюдений

Внутренние линии — Q1, медиана, Q3 (как в boxplot).

In [ ]:
# ── Violin plots ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Violin plots — форма распределения + квартили', fontsize=15, fontweight='bold')
axes = axes.flatten()

for ax, feat in zip(axes, features):
    # seaborn.violinplot принимает DataFrame напрямую:
    #   x='species'     — разбивка по оси X
    #   y=feat          — числовой признак по оси Y
    #   inner='quartile'— внутри рисует квартили (как в boxplot)
    sns.violinplot(
        data=df,
        x='species', y=feat,
        palette=PALETTE,
        inner='quartile',
        linewidth=1.5,
        ax=ax
    )
    ax.set_xlabel('Вид ириса', labelpad=8)
    ax.set_ylabel(feature_labels[feat], labelpad=8)
    ax.set_title(feature_labels[feat], pad=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.35, linestyle='--')

plt.tight_layout()
plt.show()

print('''
📌 Преимущество violin plot перед boxplot:
  Видна ФОРМА распределения, а не только 5 чисел.
  Например, у petal_length для versicolor видно, что распределение равномернее,
  чем у virginica, где данные сосредоточены в центре.
''')

### 3.5 Диаграммы рассеяния (Scatter plots)

**Scatter plot** — каждое наблюдение отображается точкой.
Цвет точки = вид ириса.

Позволяет увидеть:
- Взаимосвязь между двумя признаками
- Разделимость классов
- Выбросы

In [ ]:
# ── Scatter plots для ключевых пар признаков ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Диаграммы рассеяния', fontsize=15, fontweight='bold')

# Пары признаков для сравнения: (X-ось, Y-ось, заголовок)
pairs = [
    ('sepal_length', 'sepal_width',  'Чашелистик: длина vs ширина'),
    ('petal_length', 'petal_width',  'Лепесток: длина vs ширина'),
]

for ax, (x_feat, y_feat, title) in zip(axes, pairs):
    for color, species in zip(PALETTE, df['species'].unique()):
        subset = df[df['species'] == species]
        ax.scatter(
            subset[x_feat], subset[y_feat],
            color=color,
            label=species,
            alpha=0.75,         # полупрозрачность — видно перекрытие точек
            s=60,               # размер точки
            edgecolors='white',
            linewidths=0.6
        )
    ax.set_xlabel(feature_labels[x_feat], labelpad=8)
    ax.set_ylabel(feature_labels[y_feat], labelpad=8)
    ax.set_title(title, pad=10, fontweight='bold')
    ax.legend(title='Вид', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print('''
📌 Выводы:
  • Лепесток (правый график): три кластера чётко разделены!
    Setosa — в левом нижнем углу, сильно отделена.
    Versicolor и virginica — в правой части, частично перекрываются.
  • Чашелистик (левый): классы перемешаны — хуже для классификации.
''')

### 3.6 Матрица диаграмм рассеяния (Pair plot)

**Pair plot** — сетка scatter plots для **всех пар признаков** сразу.

- Строка и столбец = признак
- Ячейка (i, j) = scatter plot признака i vs признака j
- Диагональ = KDE-распределение каждого признака

Позволяет за **один взгляд** оценить все взаимосвязи.

In [ ]:
# ── Pair plot (матрица рассеяния) ─────────────────────────────────────────────
# Для красивых подписей переименуем столбцы
df_plot = df[features + ['species']].copy()
df_plot.columns = ['sepal\nlength', 'sepal\nwidth', 'petal\nlength', 'petal\nwidth', 'species']

# sns.pairplot — автоматически строит всю матрицу
#   hue='species'    — раскрашиваем по классу
#   diag_kind='kde'  — на диагонали: KDE (кривая плотности)
#   plot_kws         — параметры для scatter plots
#   diag_kws         — параметры для KDE
g = sns.pairplot(
    df_plot,
    hue='species',
    palette=PALETTE,
    diag_kind='kde',
    plot_kws={'alpha': 0.65, 's': 40, 'edgecolor': 'white', 'linewidth': 0.5},
    diag_kws={'fill': True, 'alpha': 0.55}
)
g.figure.suptitle('Матрица диаграмм рассеяния (Pair Plot)', y=1.02, fontsize=15, fontweight='bold')
g.figure.set_size_inches(10, 8)

plt.show()

print('''
📌 Выводы:
  • Любая пара, включающая признак лепестка (petal), хорошо разделяет классы.
  • Наилучшая пара: petal_length vs petal_width — три кластера максимально разделены.
  • Признаки чашелистика (sepal) разделяют классы хуже.
''')

### 3.7 KDE — оценка плотности распределения

**KDE (Kernel Density Estimation)** — сглаженная непрерывная версия гистограммы.

- Кривая выше → значений больше
- Не зависит от ширины «корзин» (в отличие от гистограммы)
- Позволяет точнее видеть форму распределения

In [ ]:
# ── KDE-графики ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('KDE — оценка плотности распределения по видам', fontsize=15, fontweight='bold')
axes = axes.flatten()

for ax, feat in zip(axes, features):
    for color, species in zip(PALETTE, df['species'].unique()):
        subset = df[df['species'] == species][feat]
        # fill=True  — закрашиваем область под кривой
        # alpha=0.3  — прозрачность заливки
        sns.kdeplot(
            subset, ax=ax,
            color=color, label=species,
            fill=True, alpha=0.3, linewidth=2.2
        )
    ax.set_xlabel(feature_labels[feat], labelpad=8)
    ax.set_ylabel('Плотность вероятности', labelpad=8)
    ax.set_title(feature_labels[feat], pad=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

### 3.8 Параллельные координаты

**Параллельные координаты** — способ отобразить **многомерные данные** в 2D.

- Каждый признак = вертикальная ось
- Каждое наблюдение = ломаная линия, проходящая через все оси
- Цвет = вид ириса

Пучки линий одного цвета показывают, как ведёт себя каждый класс.

In [ ]:
# ── Параллельные координаты ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

df_pc = df[features + ['species']].copy()

# parallel_coordinates ожидает DataFrame, где последний (или указанный) столбец — класс
parallel_coordinates(
    df_pc, 'species',
    color=PALETTE,
    linewidth=0.9,
    alpha=0.5,
    ax=ax
)

ax.set_title('Параллельные координаты — все признаки', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Признак', labelpad=10)
ax.set_ylabel('Значение (см)', labelpad=10)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.4, linestyle='--')
ax.legend(title='Вид', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

print('''
📌 Что видно:
  • Синие линии (setosa): резко падают на признаках лепестка → маленькие лепестки.
  • Оранжевые (versicolor) и зелёные (virginica): схожий профиль, но разный масштаб.
  • Пучки плотные → признаки внутри вида схожи (мало разброса).
''')

---
## 🔗 4. Корреляционный анализ

**Корреляция** — статистическая мера линейной взаимосвязи двух числовых переменных.

Мы используем **коэффициент Пирсона (r)**:
- `r = +1` → идеальная прямая зависимость
- `r = 0` → нет линейной связи
- `r = -1` → идеальная обратная зависимость

Правило интерпретации:
- `|r| > 0.7` — сильная корреляция
- `0.3 < |r| ≤ 0.7` — умеренная
- `|r| ≤ 0.3` — слабая / нет связи

In [ ]:
# ── 4.1 Вычисляем матрицу корреляции ─────────────────────────────────────────
# .corr() вычисляет корреляцию Пирсона для каждой пары столбцов
corr_matrix = df[features].corr()

print('Матрица коэффициентов корреляции Пирсона:')
print('(1.0 — идеальная самокорреляция на диагонали)')
corr_matrix.round(3)

In [ ]:
# ── 4.2 Тепловая карта корреляции (Heatmap) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Тепловые карты корреляции', fontsize=15, fontweight='bold')

# Короткие названия для осей
short_labels = ['sepal\nlength', 'sepal\nwidth', 'petal\nlength', 'petal\nwidth']

# ─ Левый: полная матрица, палитра RdYlGn (красный-жёлтый-зелёный) ─
# annot=True  — показываем числа в ячейках
# fmt='.2f'   — два знака после запятой
# vmin, vmax  — фиксируем шкалу от -1 до +1
# square=True — квадратные ячейки
sns.heatmap(
    corr_matrix,
    ax=axes[0],
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.8,
    square=True,
    cbar_kws={'label': 'Коэффициент Пирсона', 'shrink': 0.85},
    annot_kws={'size': 12, 'weight': 'bold'}
)
axes[0].set_title('Полная матрица (RdYlGn)', pad=12, fontweight='bold')
axes[0].set_xticklabels(short_labels, rotation=0, ha='center')
axes[0].set_yticklabels(short_labels, rotation=0)

# ─ Правый: только нижний треугольник, палитра coolwarm ─
# mask=np.triu(...) — маскируем (скрываем) верхний треугольник
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    ax=axes[1],
    mask=mask_upper,      # скрываем дубликаты (матрица симметрична)
    annot=True, fmt='.2f',
    cmap='coolwarm',      # синий = отрицательная, красный = положительная
    vmin=-1, vmax=1,
    linewidths=0.8,
    square=True,
    cbar_kws={'label': 'Коэффициент Пирсона', 'shrink': 0.85},
    annot_kws={'size': 12, 'weight': 'bold'}
)
axes[1].set_title('Нижний треугольник (coolwarm)', pad=12, fontweight='bold')
axes[1].set_xticklabels(short_labels, rotation=0, ha='center')
axes[1].set_yticklabels(short_labels, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3 Тепловые карты корреляции по видам ────────────────────────────────────
species_list = ['setosa', 'versicolor', 'virginica']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Матрицы корреляции по видам ириса\n(корреляции внутри каждого класса)', fontsize=14, fontweight='bold')

for ax, sp, color in zip(axes, species_list, PALETTE):
    # Вычисляем корреляцию только внутри одного вида
    corr_sp = df[df['species'] == sp][features].corr()
    sns.heatmap(
        corr_sp,
        ax=ax,
        annot=True, fmt='.2f',
        cmap='RdYlGn', vmin=-1, vmax=1,
        linewidths=0.6, square=True, cbar=False,
        annot_kws={'size': 10, 'weight': 'bold'}
    )
    ax.set_title(f'Iris {sp}', pad=10, fontweight='bold', color=color)
    short = ['sep.len', 'sep.wid', 'pet.len', 'pet.wid']
    ax.set_xticklabels(short, rotation=25, ha='right', fontsize=9)
    ax.set_yticklabels(short, rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

print('''
📌 Интересный факт:
  Корреляции ВНУТРИ каждого вида могут сильно отличаться от глобальных корреляций!
  Это называется «парадокс Симпсона» — когда в подгруппах наблюдается иная тенденция,
  чем в совокупности всех данных.
''')

In [ ]:
# ── 4.4 Scatter plot с линией регрессии ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Регрессионные графики: сильная и слабая корреляция', fontsize=14, fontweight='bold')

# ─ Левый: сильная корреляция — petal_length & petal_width ─
for color, sp in zip(PALETTE, species_list):
    subset = df[df['species'] == sp]
    axes[0].scatter(subset['petal_length'], subset['petal_width'],
                    color=color, label=sp, alpha=0.75, s=55, edgecolors='white', linewidths=0.5)

# np.polyfit(x, y, 1) — подбирает коэффициенты линейной функции y = mx + b
m, b = np.polyfit(df['petal_length'], df['petal_width'], 1)
x_line = np.linspace(df['petal_length'].min(), df['petal_length'].max(), 100)
axes[0].plot(x_line, m * x_line + b, 'k--', linewidth=2.2, label=f'y = {m:.2f}x + {b:.2f}')

r = df[['petal_length', 'petal_width']].corr().iloc[0, 1]   # .iloc[0,1] берёт элемент [0][1] матрицы
axes[0].set_title(f'petal_length vs petal_width\nr = {r:.3f}  (очень сильная +)', fontweight='bold')
axes[0].set_xlabel(feature_labels['petal_length'])
axes[0].set_ylabel(feature_labels['petal_width'])
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(alpha=0.3, linestyle='--')

# ─ Правый: слабая/отрицательная — sepal_width & petal_length ─
for color, sp in zip(PALETTE, species_list):
    subset = df[df['species'] == sp]
    axes[1].scatter(subset['sepal_width'], subset['petal_length'],
                    color=color, label=sp, alpha=0.75, s=55, edgecolors='white', linewidths=0.5)

m2, b2 = np.polyfit(df['sepal_width'], df['petal_length'], 1)
x_line2 = np.linspace(df['sepal_width'].min(), df['sepal_width'].max(), 100)
axes[1].plot(x_line2, m2 * x_line2 + b2, 'k--', linewidth=2.2, label=f'y = {m2:.2f}x + {b2:.2f}')

r2 = df[['sepal_width', 'petal_length']].corr().iloc[0, 1]
axes[1].set_title(f'sepal_width vs petal_length\nr = {r2:.3f}  (слабая −)', fontweight='bold')
axes[1].set_xlabel(feature_labels['sepal_width'])
axes[1].set_ylabel(feature_labels['petal_length'])
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5 Ранжирование всех попарных корреляций ─────────────────────────────────
# Собираем таблицу всех пар признаков и их корреляций
corr_pairs = []
for i, f1 in enumerate(features):
    for j, f2 in enumerate(features):
        if i < j:   # берём только верхний треугольник (без дубликатов)
            r_val = df[[f1, f2]].corr().iloc[0, 1]
            corr_pairs.append({
                'Пара признаков': f'{f1}\nvs\n{f2}',
                'r': r_val,
                'abs_r': abs(r_val)
            })

corr_df = pd.DataFrame(corr_pairs).sort_values('abs_r', ascending=False)

print('Все попарные корреляции, отсортированные по убыванию |r|:')
print(corr_df[['Пара признаков', 'r']].to_string(index=False))

print()
print('Интерпретация:')
print('  r > 0.7  → СИЛЬНАЯ положительная корреляция')
print('  r < -0.3 → слабая отрицательная')
print('  |r| < 0.3 → практически нет линейной связи')

In [ ]:
# ── Визуализация ранжирования ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

# Цвет столбца: зелёный — положительная корреляция, красный — отрицательная
colors_bar = ['#2ecc71' if r >= 0 else '#e74c3c' for r in corr_df['r']]
bars = ax.bar(
    corr_df['Пара признаков'],
    corr_df['r'],
    color=colors_bar, edgecolor='white', linewidth=1.2, width=0.55
)

# Подписываем значение r над/под каждым столбцом
for bar, r_val in zip(bars, corr_df['r']):
    offset = 0.02 if r_val >= 0 else -0.07
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset,
        f'{r_val:.3f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.axhline(0,    color='black', linewidth=0.8)
ax.axhline(0.7,  color='grey', linestyle=':', linewidth=1.5, alpha=0.8, label='|r| = 0.7 (порог сильной)')
ax.axhline(-0.7, color='grey', linestyle=':', linewidth=1.5, alpha=0.8)
ax.set_ylim(-0.55, 1.12)
ax.set_title('Ранжирование попарных корреляций Пирсона', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Коэффициент корреляции r', labelpad=8)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.35, linestyle='--')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## ✅ 5. Итоговые выводы

По результатам разведочного анализа датасета **Iris** сформулированы следующие выводы:

### 5.1 Структура и качество данных
- Датасет содержит **150 наблюдений**, **4 числовых признака**, **3 сбалансированных класса**
- Пропущенные значения **отсутствуют** — предобработка не требуется

### 5.2 Распределение признаков
- `sepal_length`, `sepal_width` — близки к **нормальному** распределению
- `petal_length`, `petal_width` — **бимодальное** распределение: явно виден *setosa* как отдельный кластер

### 5.3 Разделимость классов
- *Iris setosa* **линейно отделима** от двух других видов (достаточно порога по длине лепестка)
- *Versicolor* и *virginica* **частично перекрываются** — требуют нелинейного классификатора

### 5.4 Корреляционный анализ

| Пара признаков | r | Вывод |
|---|---|---|
| petal_length – petal_width | **≈ 0.963** | Очень сильная связь — мультиколлинеарность! |
| sepal_length – petal_length | **≈ 0.872** | Сильная |
| sepal_length – petal_width | **≈ 0.818** | Сильная |
| sepal_width – petal_length | **≈ −0.366** | Слабая отрицательная |
| sepal_width – sepal_length | **≈ −0.109** | Связи нет |

**Главный вывод:** признаки лепестка несут наибольшую информацию для классификации,
но при этом сильно скоррелированы между собой — при построении некоторых моделей
следует учитывать **мультиколлинеарность**.

In [ ]:
print('=' * 55)
print('  ✅ Лабораторная работа №1 выполнена!')
print('  Все разделы EDA реализованы:')
print('   1. Описание датасета')
print('   2. Основные характеристики (describe, dtypes, isnull)')
print('   3. Визуализация (8 типов графиков)')
print('   4. Корреляционный анализ (heatmap + scatter + ranking)')
print('=' * 55)